In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
import pickle

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/cleaned_data.csv')

In [3]:
df.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FD,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DR,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FD,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FD,19.20,Regular,0.057792,Fruits and Vegetables,182.0950,OUT010,1998,Small,Tier 3,Grocery Store,732.3800
4,Non-Edible,8.93,Low Fat,0.057792,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


- The data was collected in 2018, according to Kaggle, so we created a new feature called `Outlet_Age` by subtracting the establishment year and dropped "Outlet_Establishment_Year" feature.

In [4]:
current_year = 2018
df['Outlet_Age'] = current_year - df['Outlet_Establishment_Year']
df.drop(columns = ['Outlet_Establishment_Year'], axis= 1, inplace=True)

- Applying one-hot encoding to these features because they have fewer categories: `["Item_Identifier", "Item_Fat_Content", "Outlet_Type"]`
- Applying frequency encoding to these features because they have many categories: `["Item_Type", "Outlet_Identifier"]`
- Applying ordinal encoding to these features because an inherent order exists: `["Outlet_Size","Outlet_Location_Type"]`

In [5]:
size_map = ['Small','Medium','High']
location_map = ['Tier 1','Tier 2', 'Tier 3']
type_freq = df.Item_Type.value_counts()
identifier_freq = df.Outlet_Identifier.value_counts()

## Ordinal encoding
ordinal_enc = OrdinalEncoder(categories=[size_map, location_map])
df[['Outlet_Size','Outlet_Location_Type']] = ordinal_enc.fit_transform(df[['Outlet_Size','Outlet_Location_Type']])

# # frequency encoding
df['Item_Type'] = df['Item_Type'].map(type_freq)
df['Outlet_Identifier'] = df['Outlet_Identifier'].map(identifier_freq)

# one hot encoding
encoded_df = pd.get_dummies(df,
                              columns = ['Item_Identifier','Item_Fat_Content','Outlet_Type'],
                              dtype=int,
                              drop_first=True)

In [6]:
print(len(encoded_df.columns))

15


#### train-test split

In [7]:
features = encoded_df.drop(columns = ['Item_Outlet_Sales'], axis=1)
target = encoded_df[['Item_Outlet_Sales']]

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape) 
print(X_test.shape, y_test.shape)

(6818, 14) (6818, 1)
(1705, 14) (1705, 1)


- To ensure that every feature contributes equally to the model and features with larger value ranges do not dominate those with smaller ranges, we are applying the StandardScaler.

In [8]:
columns_to_scale = ["Item_Weight", "Item_Visibility", "Item_MRP", "Outlet_Age"]

scaler = StandardScaler()

X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])

In [9]:
# Linear Regression
model_lr = LinearRegression()
model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

print("RMSE:", root_mean_squared_error(y_test, y_pred_lr))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("R2 score:", r2_score(y_test, y_pred_lr))

RMSE: 1068.63185143498
MAE: 792.3195061509282
R2 score: 0.5798430424501102


In [10]:
## cross-validation score
scores_lr = cross_val_score(model_lr, X_train,y_train, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
print(min(-1*scores_lr))

833.415440029395


In [11]:
# random forest 
model_for = RandomForestRegressor(n_estimators=500, max_depth=5, random_state=4, max_samples=1000)

model_for.fit(X_train, y_train)

y_pred_for = model_for.predict(X_test)

print("RMSE:", root_mean_squared_error(y_test, y_pred_for))
print("MAE:", mean_absolute_error(y_test, y_pred_for))
print("R2 score:", r2_score(y_test, y_pred_for))

RMSE: 1022.5029287030881
MAE: 717.598969109742
R2 score: 0.6153334239961568


In [12]:
# cross-validation
scores = cross_val_score(model_for, X_train, y_train.values.ravel(), cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
print(min(-1*scores))

754.2485239185919


- The random forest model outperformed the linear regression model

In [13]:
# saving the random forest model
with open("../models/model_rf.pkl", 'wb') as f:
    pickle.dump(model_for, f)